In [9]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv


load_dotenv()

True

In [10]:
model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [11]:
class BlogState(TypedDict):
    title : str
    outline : str
    blog : str
    rating : str

In [12]:
def createOutline(state: BlogState) -> BlogState:
    """ Create an outline based on the provided title"""

    title = state["title"]
    prompt = f"Generate a professional outline for a blog on the topic : {title}"
    outline = model.invoke(prompt).content
    state["outline"] = outline
    return state

In [13]:
def createBlog(state : BlogState) -> BlogState:
    """Create a blog post on the basis of the provided title and outline"""

    title = state["title"]
    outline = state["outline"]
    prompt = f"On the basis of the provided title: {title} and outline: {outline}, generate a blog post."
    blog = model.invoke(prompt).content
    state["blog"] = blog
    print("blog", blog)
    return state



In [14]:
def evaluateBlog(state : BlogState) -> BlogState:
    """Evaluate the blog based the outline"""

    title = state["title"]
    outline = state["outline"]
    blog = state["blog"][0]
    prompt = f"For the topic - {title}, evaluate and give me a rating out of 10 for the attached blog. \n The outlines are : {outline} \n The blog is : {blog}"

    rating = model.invoke(prompt).content
    state["rating"] = rating
    return state


In [15]:
graph = StateGraph(BlogState)
graph.add_node('createOutline', createOutline)
graph.add_node('createBlog', createBlog)
graph.add_node('evaluateBlog', evaluateBlog)

graph.add_edge(START, 'createOutline')
graph.add_edge('createOutline', 'createBlog')
graph.add_edge('createBlog', 'evaluateBlog')
graph.add_edge('evaluateBlog', END)
graph.add_edge('createBlog', END)

workflow = graph.compile()

In [16]:
initial_state = {'title' : 'Industrial Revolution'}
final_state  = workflow.invoke(initial_state)
print(final_state["rating"])


blog [{'type': 'text', 'text': '# The Great Transformation: How the Industrial Revolution Shaped the Modern World\n\nImagine waking up in the year 1750. Your life is dictated by the rising and setting of the sun. You work the land using tools that have remained largely unchanged for centuries, and your world is confined to the few miles you can travel by foot or horse. Now, fast forward just one hundred years. The landscape is dominated by iron, soot, and the rhythmic clanking of colossal machines. You can traverse entire countries in a single day, and the goods in your home were crafted by steam-driven engines, not human hands.\n\nThis was not a gradual evolution; it was a seismic rupture in human history. The Industrial Revolution (roughly 1760–1840) was far more than a collection of new gadgets. It was a fundamental socio-economic shift that laid the very blueprint for modern global civilization.\n\n## The Catalyst: Why Britain? Why Then?\nHistory rarely pivots without a "perfect st